# Números citados no relatório

Este notebook calcula os números que aparecem **no texto** do `RELATORIO.ipynb` (os das tabelas e dos gráficos já são calculados lá). Cada seção indica em que parte do relatório o número é usado.

Os dados são os mesmos do relatório: `data/rewards/` (recompensa por episódio) e `data/q_tables/` (Q-table de cada run).

**Setup:** carrega as recompensas e as Q-tables dos 5 treinos de cada configuração.

In [1]:
from pathlib import Path

import gymnasium as gym
import numpy as np
import pandas as pd

from report_utils import load_rewards, moving_average, results_table, transfer_table

DATA_DIR = Path.cwd().parent / "data"
N_RUNS = 5

LABELS = {
    "qlearning_deterministic": "Q-Learning (determinístico)",
    "sarsa_deterministic":     "SARSA (determinístico)",
    "qlearning_stochastic":    "Q-Learning (estocástico)",
    "sarsa_stochastic":        "SARSA (estocástico)",
}

deterministic_env = gym.make("Taxi-v4")
stochastic_env = gym.make("Taxi-v4", is_rainy=True, rainy_probability=0.6)
envs = {name: stochastic_env if "stochastic" in name else deterministic_env for name in LABELS}

rewards = {name: load_rewards(DATA_DIR / "rewards", name) for name in LABELS}
q_tables = {
    name: [np.loadtxt(DATA_DIR / "q_tables" / f"{name}_run{run}.csv", delimiter=",") for run in range(1, N_RUNS + 1)]
    for name in LABELS
}
mean_curves = {name: r.mean(axis=0) for name, r in rewards.items()}

## 1. Média móvel em episódios específicos

Valor da média móvel (100 episódios) da curva média das 5 runs em alguns pontos do treino.

**Usado em:** comentários dos gráficos e seção 3 do relatório (ex.: "no episódio 300, -14 contra -44"; "no episódio 500, -40 contra -73").

In [2]:
checkpoints = [100, 200, 300, 500, 1000, 2000]
pd.DataFrame(
    {LABELS[name]: [moving_average(m)[ep - 100] for ep in checkpoints] for name, m in mean_curves.items()},
    index=[f"episódio {ep}" for ep in checkpoints],
).T.round(0)

,episódio 100,episódio 200,episódio 300,episódio 500,episódio 1000,episódio 2000
Q-Learning (determinístico),-266.0,-78.0,-14.0,2.0,5.0,5.0
SARSA (determinístico),-334.0,-101.0,-44.0,0.0,1.0,3.0
Q-Learning (estocástico),-308.0,-166.0,-105.0,-40.0,-16.0,-15.0
SARSA (estocástico),-371.0,-211.0,-148.0,-73.0,-35.0,-26.0


## 2. Patamar depois da estabilização

Faixa (mínimo e máximo) da média móvel do episódio 2000 em diante, ou seja, onde cada curva fica depois que para de subir.

**Usado em:** comentários dos gráficos (ex.: "SARSA se estabiliza entre -30 e -20"; "Q-Learning se estabiliza em ~-15").

In [3]:
rows = {}
for name, m in mean_curves.items():
    tail = moving_average(m)[2000 - 100:]
    rows[LABELS[name]] = {"Mínimo da MM": tail.min(), "Máximo da MM": tail.max(), "Média da MM": tail.mean()}
pd.DataFrame(rows).T.round(1)

,Mínimo da MM,Máximo da MM,Média da MM
Q-Learning (determinístico),4.5,6.2,5.2
SARSA (determinístico),0.6,4.5,3.2
Q-Learning (estocástico),-19.1,-11.1,-14.8
SARSA (estocástico),-32.2,-18.2,-22.6


## 3. Episódios presos no fim do treino

Porcentagem dos últimos 1000 episódios de treino com recompensa ≤ -200, isto é, que bateram o limite de 200 passos sem entregar.

**Usado em:** comentário do primeiro gráfico ("No fim do treino, nenhum episódio fica preso até o limite de 200 passos").

In [4]:
pd.Series(
    {LABELS[name]: 100 * (r[:, -1000:] <= -200).mean() for name, r in rewards.items()},
    name="% de episódios truncados (últ. 1000)",
).round(2)

Q-Learning (determinístico)    0.00
SARSA (determinístico)         0.00
Q-Learning (estocástico)       0.00
SARSA (estocástico)            0.16
Name: % de episódios truncados (últ. 1000), dtype: float64

## 4. Efeito da chuva em cada algoritmo

Compara o determinístico com o estocástico usando os mesmos números da tabela do relatório: queda da recompensa, aumento dos passos por entrega e do desvio padrão.

**Usado em:** seção 4 do relatório (ex.: "cai ~19 pontos no Q-Learning"; "passos sobem de ~13 a 14 para ~27 a 30"; "desvio sobe de ~3 para ~11 a 16").

In [5]:
table = results_table(rewards, q_tables, envs, LABELS)
reward_col = table.columns[0]

rows = {}
for algorithm, label in [("qlearning", "Q-Learning"), ("sarsa", "SARSA")]:
    det = table.loc[LABELS[f"{algorithm}_deterministic"]]
    rain = table.loc[LABELS[f"{algorithm}_stochastic"]]
    rows[label] = {
        "Recompensa: det → chuva": f"{det[reward_col]:.1f} → {rain[reward_col]:.1f}",
        "Queda da recompensa": det[reward_col] - rain[reward_col],
        "Timesteps: det → chuva": f"{det['Timesteps médios']:.1f} → {rain['Timesteps médios']:.1f}",
        "Aumento dos timesteps (×)": rain["Timesteps médios"] / det["Timesteps médios"],
        "Desvio dos timesteps: det → chuva": f"{det['Desvio padrão dos timesteps']:.1f} → {rain['Desvio padrão dos timesteps']:.1f}",
        "Sucesso: det → chuva (%)": f"{det['Taxa de sucesso (%)']:.1f} → {rain['Taxa de sucesso (%)']:.1f}",
    }
pd.DataFrame(rows).T.round(2)

,Recompensa: det → chuva,Queda da recompensa,Timesteps: det → chuva,Aumento dos timesteps (×),Desvio dos timesteps: det → chuva,Sucesso: det → chuva (%)
Q-Learning,5.1 → -14.3,19.37,13.4 → 27.1,2.025449,2.9 → 11.4,100.0 → 83.8
SARSA,3.2 → -20.9,24.06,13.9 → 30.0,2.162221,3.3 → 15.5,81.8 → 83.0


## 5. Política determinística testada na chuva

Os mesmos números da tabela da Parte 4 do relatório, repetidos aqui para conferir as frases do texto.

**Usado em:** discussão da seção "Política determinística testada na chuva" (ex.: "entrega em ~98% dos episódios"; "98% contra 84%").

In [6]:
transfer_table(q_tables, deterministic_env, stochastic_env)

Timesteps médios  \
Algoritmo  Cenário                                                          
Q-Learning treino determinístico → teste determinístico             13.36   
           treino determinístico → teste com chuva                  26.57   
           treino com chuva → teste com chuva                       27.06   
SARSA      treino determinístico → teste determinístico             13.87   
           treino determinístico → teste com chuva                  28.31   
           treino com chuva → teste com chuva                       29.99   

                                                         Desvio padrão dos timesteps  \
Algoritmo  Cenário                                                                     
Q-Learning treino determinístico → teste determinístico                         2.88   
           treino determinístico → teste com chuva                             10.64   
           treino com chuva → teste com chuva                                  11.38   
SARSA      treino determinístico → teste determinístico                         3.27   
           treino determinístico → teste com chuva                             12.47   
           treino com chuva → teste com chuva                                  15.50   

                                                         Taxa de sucesso (%)  \
Algoritmo  Cenário                                                             
Q-Learning treino determinístico → teste determinístico                100.0   
           treino determinístico → teste com chuva                      97.8   
           treino com chuva → teste com chuva                           83.8   
SARSA      treino determinístico → teste determinístico                 81.8   
           treino determinístico → teste com chuva                      83.6   
           treino com chuva → teste com chuva                           83.0   

                                                         Desvio da taxa de sucesso entre runs  
Algoritmo  Cenário                                                                             
Q-Learning treino determinístico → teste determinístico                                  0.00  
           treino determinístico → teste com chuva                                       2.77  
           treino com chuva → teste com chuva                                           19.84  
SARSA      treino determinístico → teste determinístico                                 14.04  
           treino determinístico → teste com chuva                                      14.91  
           treino com chuva → teste com chuva                                           13.82

## 6. Contas teóricas de γ e ε

Não dependem dos dados: são contas a partir dos hiperparâmetros (γ = 0.99, ε de 0.2 a 0.05 com decaimento de 0.99).
- **Valor descontado da entrega**: quanto vale o +20 visto a *d* passos de distância, ou seja, 20 · γ^*d*.
- **Horizonte efetivo**: 1 / (1 − γ), quantos passos à frente o agente "enxerga".
- **Episódios até ε chegar ao mínimo**: menor *n* com 0.2 · 0.99^*n* ≤ 0.05.

**Usado em:** seção 3 do relatório ("o +20 ainda vale ~18 a 10 passos") e na tabela de hiperparâmetros.

In [7]:
GAMMA = 0.99

distances = [5, 10, 15, 30]
display(pd.Series({f"{d} passos": 20 * GAMMA ** d for d in distances}, name="Valor do +20 da entrega").round(2))

print(f"Horizonte efetivo com γ = {GAMMA}: {1 / (1 - GAMMA):.0f} passos")

epsilon, n = 0.2, 0
while epsilon > 0.05:
    epsilon *= 0.99
    n += 1
print(f"ε chega a 0.05 depois de {n} episódios")

5 passos     19.02
10 passos    18.09
15 passos    17.20
30 passos    14.79
Name: Valor do +20 da entrega, dtype: float64

Horizonte efetivo com γ = 0.99: 100 passos
ε chega a 0.05 depois de 138 episódios
